In [41]:
import polars as pl
from scipy.stats import norm
from plotly.offline import init_notebook_mode

init_notebook_mode(connected=True)
import plotly.express as px
import plotly.graph_objects as go

In [42]:
budget = (
    pl.concat(
        [
            pl.read_json("budget2024.json", infer_schema_length=None),
            pl.read_json("budget2025.json", infer_schema_length=None),
        ]
    )
    .unnest("data")
    .unnest("budgetVsActualV2")
)

In [43]:
def extract_cashflow(cashflow: pl.Series):
    return (
        cashflow.explode()
        .struct.unnest()
        .select("subAccounts", headline_category="category")
        .explode("subAccounts")
        .unnest("subAccounts")
        .select(
            "headline_category",
            "category",
            pl.col("annual").struct.field("transactions"),
        )
        .explode("transactions")
        .unnest("transactions")
        .with_columns(pl.col("date").str.to_date())
        .drop(["transactionType", "__typename"])
        .drop_nulls()
    )


cashflow = pl.concat(
    [
        extract_cashflow(budget["incomes"]).with_columns(is_income=pl.lit(True)),
        extract_cashflow(budget["expenses"]).with_columns(is_income=pl.lit(False)),
    ]
)

# Simulate 13 months because we need to see what happens this December to know how much budget we start next year with.
MONTHS_TO_SIMULATE = 13
SIMULATIONS = 10000
UNPAID_BILLS = 13597.12
# Based on the "Operation" account pulled from Daisy dashboard on 2025-11-18
STARTING_BALANCE = 15711.40 - UNPAID_BILLS
# Simulate budget increases between 1% - 5% in increments of 1 percentage point.
BUDGET_INCREASES = list(
    round(budget_increase / 100.0, 2) for budget_increase in range(-3, 7)
)
cashflow_by_month_by_increase = [
    (
        budget_increase,
        cashflow.with_columns(
            pl.col("amount")
            * pl.when(pl.col("is_income")).then(1 + budget_increase).otherwise(-1)
        )
        .group_by(
            year=pl.col("date").dt.year(),
            month=pl.col("date").dt.month(),
        )
        .agg(pl.col("amount").sum()),
    )
    for budget_increase in BUDGET_INCREASES
]

simulations = pl.concat(
    pl.collect_all(
        (
            cashflow_by_month.lazy()
            .select(
                pl.col("amount").sample(MONTHS_TO_SIMULATE, shuffle=True).cum_sum()
                + pl.lit(STARTING_BALANCE),
            )
            .with_columns(
                simulation_id=pl.lit(simulation_id),
                budget_increase=pl.lit(budget_increase),
            )
            .with_row_index("month")
            for (
                budget_increase,
                cashflow_by_month,
            ) in cashflow_by_month_by_increase
            for simulation_id in range(0, SIMULATIONS)
        )
    )
)

In [44]:
simulations_by_month = (
    simulations.group_by("month", "budget_increase")
    .agg(
        amount_average=pl.col("amount").mean(),
        amount_min=pl.col("amount").min(),
        amount_max=pl.col("amount").max(),
    )
    .sort("budget_increase", "month")
)

fig = go.Figure(
    [
        chart
        for (budget_increase, sims) in (
            (
                budget_increase,
                simulations_by_month.filter(
                    pl.col("budget_increase") == budget_increase
                ),
            )
            for budget_increase in BUDGET_INCREASES
        )
        for chart in [
            go.Scatter(x=sims["month"], y=sims["amount_average"], name=budget_increase),
            # go.Scatter(
            #     x=pl.concat([sims["month"], sims["month"].reverse()]),
            #     y=pl.concat([sims["amount_min"], sims["amount_max"].reverse()]),
            #     fill="toself",
            # ),
        ]
    ],
    layout=go.Layout(legend=go.layout.Legend(title="budget_increase")),
)
fig.show()

In [45]:
simulation_mins = simulations.group_by("budget_increase", "simulation_id").agg(
    pl.col("amount").min()
)

In [46]:
fig = px.pie(
    simulation_mins.group_by(
        "budget_increase",
        ruinous=pl.col("amount") < 0,
    )
    .len("simulation_count")
    .with_columns(
        ruinous=pl.when("ruinous")
        .then(pl.lit("Special Assessment"))
        .otherwise(pl.lit("Safe"))
    )
    .sort("budget_increase"),
    names="ruinous",
    values="simulation_count",
    facet_col="budget_increase",
    facet_col_wrap=3,
    title="Likelihood of Special Assessment",
    color_discrete_sequence=["#4B08AF", "#32965D"],
)
fig.show(renderer="notebook_connected")

# Special Assessment
95% confidence that the special assessment--if there is one--will be less than `amount`

In [47]:
# Inflation estimation from https://www.federalreserve.gov/monetarypolicy/files/fomcprojtabl20250917.pdf
INFLATION = 0.026
simulation_mins.filter(pl.col("amount") < 0).sort("budget_increase").with_columns(
    -pl.col("amount")
).group_by("budget_increase").agg(
    pl.col("amount").mean() + pl.col("amount").std() * norm.ppf(0.95)
).with_columns(
    amount_with_inflation=pl.col("amount") * (1 + INFLATION)
)

budget_increase,amount,amount_with_inflation
f64,f64,f64
-0.03,30541.58591,31335.667144
-0.02,29147.740253,29905.5815
-0.01,27667.851376,28387.215511
0.0,25647.030532,26313.853326
0.01,24563.111199,25201.75209
0.02,23146.519597,23748.329107
0.03,22243.243602,22821.567936
0.04,20987.514177,21533.189545
0.05,19923.80068,20441.819497


In [48]:
fig = px.bar(
    extract_cashflow(budget["expenses"]).with_columns(
        chart_date=pl.date(pl.col("date").dt.year(), pl.col("date").dt.month(), 1)
    ),
    x="chart_date",
    y="amount",
    color="category",
)
fig.show(renderer="notebook_connected")

In [49]:
NON_RECURRING_REPAIRS = "Non-Recurring-  Repairs"
monthly_expenses = (
    budget["expenses"]
    .explode()
    .struct.unnest()
    .select("category", "monthlyBreakdown")
    .explode("monthlyBreakdown")
    .unnest("monthlyBreakdown")
    .with_columns(
        pl.col("category").cast(
            pl.Enum(
                [
                    "Utilities",
                    "Payroll Expenses",
                    "Compliance & Monitoring",
                    "Insurance",
                    NON_RECURRING_REPAIRS,
                    "Building Operating Expenses",
                    "Taxes",
                    "Reserve Contributions/Transfers",
                    "Administrative Expenses",
                ]
            )
        )
    )
)
nrr_actual = (
    monthly_expenses.filter(pl.col("category") != NON_RECURRING_REPAIRS)
    .group_by(["month", "year"])
    .agg(pl.col("actual").sum())["actual"]
)
projected_monthly_budget = nrr_actual.mean() + nrr_actual.std() * norm.ppf(0.95)
CURRENT_MONTHLY_BUDGET = 19797.62
projected_increase = (
    projected_monthly_budget - CURRENT_MONTHLY_BUDGET
) / CURRENT_MONTHLY_BUDGET

In [50]:
projected_monthly_budget

np.float64(33004.77886008375)

In [51]:
projected_increase

np.float64(0.6671084130356959)